# Sparse Tensors as Matrices (sptenmat)
```
Copyright 2025 National Technology & Engineering Solutions of Sandia,
LLC (NTESS). Under the terms of Contract DE-NA0003525 with NTESS, the
U.S. Government retains certain rights in this software.
```

We show how to convert an `sptensor` to a matrix stored in _coordinate_ format with extra information so that is can be convertered back to an `sptensor`.

[API reference](../sptenmat.rst)

In [ ]:
from __future__ import annotations

import numpy as np

import pyttb as ttb

## Creating instances

### Creating an `sptenmat` (sparse tensor as sparse matrix) object
A sparse tensor can be converted to a sparse matrix, with row and column indices stored explicitly.

First, we crease a sparse tensor to be converted.

In [ ]:
np.random.seed(0)  # Random seed for reproducibility
X = ttb.sptenrand((10, 10, 10, 10), nonzeros=10)
X

Similar options as `tenmat` are available for `sptenmat`.

In [ ]:
A = X.to_sptenmat(np.array([0]))  # Mode-0 matricization
A

In [ ]:
A = X.to_sptenmat(np.array([1, 2]))  # Multiple modes mapped to rows.
A

In [ ]:
A = X.to_sptenmat(cdims=np.array([1, 2]))  # Specify column dimensions.
A

In [ ]:
A = X.to_sptenmat(np.arange(4))  # All modes mapped to rows, i.e., vectorize.
A

In [ ]:
A = X.to_sptenmat(np.array([1]))  # By default, columns are ordered as [0, 2, 3]
A

In [ ]:
A = X.to_sptenmat(np.array([1]), np.array([3, 0, 2]))  # Specify explicit ordering
A

In [ ]:
A = X.to_sptenmat(np.array([1]), cdims_cyclic="fc")  # Forward cyclic column ordering
A

In [ ]:
A = X.to_sptenmat(np.array([1]), cdims_cyclic="bc")  # Backward cyclic column ordering
A

### Creating an `sptenmat` from its constituent parts

In [ ]:
B = ttb.sptenmat(A.subs, A.vals, A.rdims, A.cdims, A.tshape)  # Effectively copies A
B

### Creating an `sptenmat` with no nonzeros

In [ ]:
A = ttb.sptenmat(rdims=A.rdims, cdims=A.cdims, tshape=A.tshape)  # An empty sptenmat
A

### Creating an empty sptenmat

In [ ]:
A = ttb.sptenmat()  # A really empty sptenmat
A

## Key properties

### Constituent parts of an `sptenmat`

In [ ]:
np.random.seed(0)  # Random seed for reproducibility
X = ttb.sptenrand((10, 10, 10, 10), nonzeros=10)
A = X.to_sptenmat(np.array([1]), cdims_cyclic="bc")
A.subs  # Subscripts of the nonzeros.

In [ ]:
A.vals  # Corresponding nonzero values.

In [ ]:
A.tshape  # Shape of the original tensor.

In [ ]:
A.rdims  # Dimensions that were mapped to the rows.

In [ ]:
A.cdims  # Dimensions that were mapped to the columns.

### Use `shape` and `tshape` for the dimensions of an `sptenmat`

In [ ]:
print(f"Matrix shape: {A.shape}\nOriginal tensor shape: {A.tshape}")

## Converting to other types

### Use `double` to convert an `sptenmat` to a SciPy COO Matrix

In [ ]:
X = ttb.sptenrand((10, 10, 10, 10), nonzeros=10)  # Create sptensor
A = X.to_sptenmat(np.array([0]))  # Convert to an sptenmat
A

In [ ]:
B = A.double()  # Convert to scipy
B

### Use `full` to convert an `sptenmat` to a `tenmat`

In [ ]:
B = ttb.sptenrand((3, 3, 3), nonzeros=3).to_sptenmat(np.array([0]))
B

In [ ]:
C = B.full()
C

### Use `to_sptensor` to convert an `sptenmat` to an `sptensor`.

In [ ]:
Y = B.to_sptensor()
Y

## Core methods

### Subscripted assignment for an `sptenmat`

In [ ]:
A[0:2, 0:2] = 1
A

### Use `nnz` to count the nonzeros of an `sptenmat`

In [ ]:
S_m = ttb.sptensor(
    np.array([[0, 0, 0], [1, 1, 1], [1, 2, 0]]),
    np.array([[1.0], [2.0], [3.0]]),
    shape=(2, 3, 2),
)
A_m = S_m.to_sptenmat(np.array([0]))
A_m.nnz

### Use `from_array` to create an `sptenmat` from a matrix
The original tensor shape, `tshape`, and the dimensions mapped to the rows and the columns are given so the `sptenmat` can be converted back to an `sptensor`.

In [ ]:
from scipy import sparse

B_m = sparse.coo_matrix(np.array([[1.0, 0.0, 0.0, 2.0], [0.0, 3.0, 0.0, 0.0]]))
C_m = ttb.sptenmat.from_array(B_m, np.array([0]), np.array([1, 2]), tshape=(2, 2, 2))
C_m.to_sptensor()

### Use `copy` to make an independent copy of an `sptenmat`

In [ ]:
S_m = ttb.sptensor(
    np.array([[0, 0, 0], [1, 1, 1], [1, 2, 0]]),
    np.array([[1.0], [2.0], [3.0]]),
    shape=(2, 3, 2),
)
A_m = S_m.to_sptenmat(np.array([0]))
B_m = A_m.copy()
print(B_m is A_m)
B_m.isequal(A_m)

### Use `isequal` to test if two `sptenmat`s are equal

In [ ]:
S_m = ttb.sptensor(
    np.array([[0, 0, 0], [1, 1, 1], [1, 2, 0]]),
    np.array([[1.0], [2.0], [3.0]]),
    shape=(2, 3, 2),
)
A_m = S_m.to_sptenmat(np.array([0]))
print(A_m.isequal(A_m.copy()))
A_m.isequal(S_m.to_sptenmat(np.array([1])))

## Operators

### Basic operations for `sptenmat`

In [ ]:
A.norm()  # Norm of the matrix.

In [ ]:
+A  # Positive version of matrix (no change)

In [ ]:
-A  # Negative version of matrix